# brepgraph demo

Turn a B-rep CAD file into an ML-ready graph. Run the cells top to bottom.
Section 1 builds sample parts so everything works out of the box; section 2 can load your own STEP files.

In [ ]:
import sys, tempfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import brepgraph as bg
print("brepgraph", bg.__version__)

# the sample-part builders live in the repo's tests folder (they use OCP directly)
sys.path.insert(0, str(Path.cwd().parent / "tests"))
import fixtures as F

work = Path(tempfile.mkdtemp())
SAMPLES = {
    "box": F.box(10, 20, 30),
    "cylinder": F.cylinder(5, 10),
    "pocket_box": F.pocket_box(),
    "filleted_box": F.filleted_box(),
    "plate_with_hole": F.plate_with_hole(),
}
paths = {name: F.write_step(shape, work / f"{name}.step") for name, shape in SAMPLES.items()}
paths

## 1. Load a part and inspect the graph

In [ ]:
g = bg.load(paths["pocket_box"])
g            # lists every feature with shape and dtype

In [ ]:
print("faces:", g.num_faces, " edges:", g.num_edges)
print("edge_index (which two faces each edge joins):\n", g.edge_index[:, :8], "...")
print("dropped edges:", g.meta["dropped_edges"])
print("features used:", [(f["name"], f["version"]) for f in g.meta["features"]["node"]])

### Face and edge features
Surface type codes: 0 plane, 1 cylinder, 2 cone, 3 sphere, 4 torus, 5 Bezier, 6 B-spline, 7 revolution, 8 extrusion, 9 offset, 10 other.
Convexity: 0 concave, 1 convex, 2 smooth.

In [ ]:
faces = pd.DataFrame({
    "surface_type": g.node_feats["surface_type"],
    "area": g.node_feats["area"].round(2),
    "cx": g.node_feats["centroid"][:, 0].round(2),
    "cy": g.node_feats["centroid"][:, 1].round(2),
    "cz": g.node_feats["centroid"][:, 2].round(2),
})
edges = pd.DataFrame({
    "face_a": g.edge_index[0], "face_b": g.edge_index[1],
    "curve_type": g.edge_feats["curve_type"],
    "length": g.edge_feats["length"].round(2),
    "convexity": g.edge_feats["convexity"],
    "dihedral_deg": np.degrees(g.edge_feats["dihedral_angle"]).round(1),
})
display(faces); display(edges.head(12))
print("concave edges:", int((edges.convexity == 0).sum()), " convex:", int((edges.convexity == 1).sum()))

## 2. Parts to explore
By default the sample parts from section 1 are used. To explore your own dataset, set the environment variable
`BREPGRAPH_DEMO_DIR` to a folder of `.step`/`.stp` files before starting Jupyter: the first five single-solid files are
loaded. Multi-solid files (assemblies) are rejected by design and skipped.

In [ ]:
import os, time

DATA_DIR = os.environ.get("BREPGRAPH_DEMO_DIR")
parts = {}
if DATA_DIR:
    for f in sorted(Path(DATA_DIR).rglob("*")):
        if f.suffix.lower() in (".step", ".stp"):
            try:
                parts[f.stem] = bg.load(f)
            except bg.BrepGraphError:
                continue
            if len(parts) == 5:
                break
if not parts:
    parts = {name: bg.load(p) for name, p in paths.items()}

for name, gg in parts.items():
    print(f"{name:20s} faces={gg.num_faces:4d} edges={gg.num_edges:4d}")

g = next(iter(parts.values()))   # the plots below use this part
g

### Your own file (optional)

In [ ]:
MY_FILE = None   # e.g. "C:/Users/you/Downloads/part.step"
if MY_FILE:
    try:
        g = bg.load(MY_FILE); print(g)
    except bg.BrepGraphError as e:
        print(type(e).__name__, "->", e)

### Surface and convexity mix per part

In [ ]:
names = {0: "plane", 1: "cylinder", 2: "cone", 3: "sphere", 4: "torus", 6: "bspline"}
rows = []
for cat, gg in parts.items():
    st = pd.Series(gg.node_feats["surface_type"]).map(lambda k: names.get(k, f"type{k}")).value_counts()
    rows.append({"part": cat, "faces": gg.num_faces, "edges": gg.num_edges, **st.to_dict(),
                 "concave": int((gg.edge_feats["convexity"] == 0).sum()),
                 "convex": int((gg.edge_feats["convexity"] == 1).sum()),
                 "smooth": int((gg.edge_feats["convexity"] == 2).sum())})
pd.DataFrame(rows).fillna(0).set_index("part")

## 3. Visualize the selected part

In [ ]:
grid = g.node_feats["uv_grid"]                      # (F, 7, n, n)
fig = plt.figure(figsize=(13, 5))

ax = fig.add_subplot(1, 2, 1, projection="3d")
cmap = plt.get_cmap("tab20")
for f in range(g.num_faces):
    xyz = grid[f, :3].reshape(3, -1)
    inside = grid[f, 6].reshape(-1) > 0.5            # mask: sample lies in the trimmed face
    ax.scatter(*xyz[:, inside], s=4, color=cmap(f % 20))
    ax.scatter(*xyz[:, ~inside], s=4, color="lightgray", alpha=0.3)
ax.set_title("UV samples (gray = outside trim, masked out)")

ax = fig.add_subplot(1, 2, 2, projection="3d")
c = g.node_feats["centroid"]
ax.scatter(*c.T, s=40, color="black")
conv_colors = {0: "red", 1: "green", 2: "blue"}
for k, (a, b) in enumerate(g.edge_index.T):
    ax.plot(*np.stack([c[a], c[b]]).T, color=conv_colors[int(g.edge_feats["convexity"][k])], lw=1)
ax.set_title("face graph: green convex, red concave, blue smooth")
plt.tight_layout(); plt.show()

### Normals on one face (outward-pointing)

In [ ]:
f = int(np.argmax(g.node_feats["area"]))
xyz = grid[f, :3].reshape(3, -1); nrm = grid[f, 3:6].reshape(3, -1)
inside = grid[f, 6].reshape(-1) > 0.5
ax = plt.figure(figsize=(5, 5)).add_subplot(projection="3d")
ax.quiver(*xyz[:, inside], *nrm[:, inside], length=2, normalize=True, color="C0")
ax.set_title(f"face {f}: surface_type={g.node_feats['surface_type'][f]}")
plt.show()
print("unit length where mask=1:", np.allclose(np.linalg.norm(nrm[:, inside], axis=0), 1, atol=1e-5))

## 4. Compare all sample parts

In [ ]:
rows = []
for name, p in paths.items():
    gg = bg.load(p)
    rows.append({"part": name, "faces": gg.num_faces, "edges": gg.num_edges,
                 "total_area": float(gg.node_feats["area"].sum().round(1)),
                 "concave": int((gg.edge_feats["convexity"] == 0).sum()),
                 "smooth": int((gg.edge_feats["convexity"] == 2).sum())})
pd.DataFrame(rows)

## 5. Options: grid size, feature selection, normalization

In [ ]:
g16 = bg.load(paths["box"], grid_size=16)
print("grid_size=16 ->", g16.node_feats["uv_grid"].shape, g16.edge_feats["u_grid"].shape)

fs = bg.FeatureSet(nodes=["surface_type", "area"], edges=["length", "convexity"])
print("light feature set ->", list(bg.load(paths["box"], fs).node_feats))

gn = bg.load(paths["box"], normalize=True)
print("normalized bbox:", np.round(gn.meta["bbox"], 3))

## 6. Add your own feature

In [ ]:
@bg.node_feature("is_planar", dtype="int64", shape=())
def is_planar(ctx, i):
    return int(ctx.surface(i).GetType() == 0)

gp = bg.load(paths["filleted_box"], bg.FeatureSet(nodes=["area", "is_planar"], edges=["length"]))
print(gp.node_feats["is_planar"], "<- the 0 is the fillet")

## 7. Interactive 3D view (`bg.viz`, needs `pip install brepgraph[viz]`)
`show` re-loads the source file recorded in the graph, tessellates it, and colours each face by a node feature.
Integer features get a discrete palette + legend; float features get a colour bar. Edges are coloured by convexity.
Drag to rotate; click legend entries to hide/show categories.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

part = g
bg.viz.show(part, "surface_type").show()

In [ ]:
bg.viz.show(part, "area", edges=False).show()                       # continuous colours
bg.viz.show(part, "surface_type", edges=False, graph=True).show()   # + face-adjacency graph (nodes numbered by face id)

### Single-face debug view: UV samples inside/outside the trim, with outward normals

In [ ]:
f = int(np.argmax(part.node_feats["area"]))
bg.viz.show_face(part, f).show()

### Graph view: the part's face graph on its own
`show_graph` draws one node per face and one edge per shared B-rep edge, without the 3-D geometry. Node size follows `area`,
node colour follows `color_by`, edge colour is convexity (red concave, green convex, blue smooth). Hover a node or an edge
midpoint for all its features. Layouts: `"spring"` (2-D), `"spring3d"`, and `"centroid"` (nodes sit at the face centroids,
so the graph keeps the part's shape). It needs only the graph, not the source file.

In [ ]:
bg.viz.show_graph(part).show()                                  # 2-D force-directed layout

In [ ]:
bg.viz.show_graph(part, "area", layout="centroid").show()       # 3-D, nodes at face centroids, coloured by area
bg.viz.show_graph(list(parts.values())[-1], "surface_type", layout="spring3d").show()

## 8. BrepMFR-style features (face-pair histograms)
The `brepmfr` preset follows Zhang et al. (CAGD 2024): 5x5 grids, loop / adjacency counts, and **face-pair** matrices:
`shortest_path` (edges between two faces), `d2` (distribution of the distance between random points on the two faces, as a
fraction of the part's bounding-box diagonal) and `a3` (angle distribution). Histograms have 64 bins and sum to 1.

In [ ]:
gm = bg.load(part.meta["source_path"], "brepmfr")
print(gm)

In [ ]:
sp, d2 = gm.pair_feats["shortest_path"], gm.pair_feats["d2"]
centres = (np.arange(64) + 0.5) / 64
mean_d2 = d2 @ centres                       # mean normalised distance between each pair of faces

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
im = ax[0].imshow(sp, cmap="viridis"); ax[0].set_title("shortest path (edges)"); plt.colorbar(im, ax=ax[0])
im = ax[1].imshow(mean_d2, cmap="magma"); ax[1].set_title("mean D2 (fraction of bbox diagonal)"); plt.colorbar(im, ax=ax[1])
i, j = np.unravel_index(np.argmax(mean_d2), mean_d2.shape)
ax[2].bar(range(64), d2[i, j]); ax[2].set_title(f"D2 histogram of the most distant pair (faces {i}, {j})")
ax[2].set_xlabel("bin (distance / bbox diagonal)")
plt.tight_layout(); plt.show()
print("D2 vs A3 for that pair: D2 mean", round(float(d2[i, j] @ centres), 3), " A3 mean", round(float(gm.pair_feats["a3"][i, j] @ centres), 3))

In [ ]:
# Pair features are symmetric and each histogram sums to 1; unreachable pairs (a part whose faces form
# disconnected groups, e.g. one with an internal cavity) get shortest_path = -1.
print("symmetric:", np.allclose(d2, d2.transpose(1, 0, 2)), " sums to 1:", np.allclose(d2.sum(-1), 1, atol=1e-5))
print("unreachable pairs in this part:", int((sp < 0).sum()))
print("size of d2:", d2.nbytes / 1e6, "MB  (grows with faces squared)")

## 9. Datasets: multi-body files and parallel loading
`bg.load` expects one solid. `bg.load_all` returns one graph per solid, and `bg.load_many` loads many files in
worker processes: a file that fails is reported in its result instead of stopping the batch. From the command line,
`brepgraph batch data/ -o graphs/ --all-solids` does the same for a whole folder and writes a `report.json`.

In [ ]:
assembly = F.write_step(F.compound(F.box(), F.cylinder(), F.pocket_box()), work / "assembly.step")
for g in bg.load_all(assembly):
    print(f"solid {g.meta['solid_index']} of {g.meta['num_solids']}: {g.num_faces} faces, {g.num_edges} edges")

In [ ]:
broken = work / "broken.step"
broken.write_text("not a step file")
results = bg.load_many([*paths.values(), assembly, broken], workers=2, all_solids=True)
for r in results:
    status = f"{len(r.graphs)} graph(s)" if r.ok else f"failed: {r.error[:60]}"
    print(f"{Path(r.source).name:22s} {status}")

## 10. Save, reload, export

In [ ]:
out = work / "pocket.npz"
g = bg.load(paths["pocket_box"])
g.save(out)
g2 = bg.BrepGraph.load(out)
print("round trip equal:", all(np.array_equal(g.node_feats[k], g2.node_feats[k]) for k in g.node_feats))

from brepgraph.export import transforms
print("one-hot convexity:", transforms.one_hot(g, "convexity").shape)

for name, convert in [("PyG", lambda: g.to_pyg(x=("area", "centroid"))), ("DGL", lambda: g.to_dgl())]:
    try:
        print(name, "->", convert())
    except bg.MissingExtraError as e:
        print(name, "adapter needs an extra:", e)
    except Exception as e:  # e.g. a DGL build without native libraries
        print(name, "installed but unusable here:", type(e).__name__)

## 11. Errors are catchable

In [ ]:
bad = work / "fake.step"; bad.write_text("not a step file")
for label, src in [("bad file", bad), ("two solids", None)]:
    try:
        bg.load(src if src else F.two_boxes())
    except bg.BrepGraphError as e:
        print(f"{label}: {type(e).__name__}: {e}")